<a href="https://colab.research.google.com/github/tioluwaniiyin123/ELEN-5301/blob/Profiler/torch.profiler_CPU.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
pip install torch torchvision

In [12]:


# EOS PyTorch version
!wget https://raw.githubusercontent.com/dionhaefner/pyhpc-benchmarks/master/benchmarks/equation_of_state/eos_pytorch.py


--2025-10-19 19:29:30--  https://raw.githubusercontent.com/dionhaefner/pyhpc-benchmarks/master/benchmarks/equation_of_state/eos_pytorch.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 7318 (7.1K) [text/plain]
Saving to: ‘eos_pytorch.py.1’

eos_pytorch.py.1    100%[===================>]   7.15K  --.-KB/s    in 0s      

2025-10-19 19:29:30 (60.9 MB/s) - ‘eos_pytorch.py.1’ saved [7318/7318]



In [13]:
!python eos_pytorch.py

In [14]:
import torch
import torchvision.models as models
from torch.profiler import profile, ProfilerActivity, record_function
from eos_pytorch import gsw_dHdT

In [15]:
if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"
    print(" CUDA not available — running on CPU only.")

model = models.resnet18().to(device)
inputs = torch.randn(5, 3, 224, 224).to(device)

 CUDA not available — running on CPU only.


In [16]:
import torch
from torch.profiler import profile, ProfilerActivity, record_function
from torchvision import models
from eos_pytorch import gsw_dHdT

# Choose device (CPU or GPU if available)
if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"
    print(" CUDA not available — running on CPU only.")

# Build activities list
activities = [ProfilerActivity.CPU]
if device == "cuda":
    activities += [ProfilerActivity.CUDA]

# Sort key for profiler output
sort_by_keyword = "cuda_memory_usage" if device == "cuda" else "cpu_memory_usage"

# Start profiling
with profile(
    activities=activities,
    record_shapes=True,
    profile_memory=True,
    with_stack=True,
    on_trace_ready=torch.profiler.tensorboard_trace_handler(f'./log/{device}')
) as prof:
    with record_function("model_inference"):
        output = model(inputs)
        # Example call to your custom function
        gsw_dHdT(inputs, torch.randn_like(inputs), torch.randn_like(inputs))
    prof.step()

# Print profiling summary
print(prof.key_averages().table(sort_by=sort_by_keyword, row_limit=10))


 CUDA not available — running on CPU only.
---------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                             Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg       CPU Mem  Self CPU Mem    # of Calls  
---------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                        aten::mul        24.26%     289.475ms        24.59%     293.373ms       1.504ms     143.55 MB     143.55 MB           195  
                      aten::empty         0.11%       1.357ms         0.11%       1.357ms       6.787us      94.86 MB      94.86 MB           200  
                 aten::empty_like         0.02%     179.704us         0.05%     553.949us      25.180us      53.12 MB           0 B            22  
                 aten::batch_norm         0.02%     186.080us        